<a href="https://colab.research.google.com/github/vituhaa/Healthy-Posture/blob/project/movenet_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Использование MoveNet: https://www.tensorflow.org/hub/tutorials/movenet?hl=ru

Чистовик:

In [ ]:
! pip install -q imageio
! pip install -q git+https://github.com/tensorflow/docs
! pip install matplotlib==3.9.0

In [1]:
import tensorflow as tf
import tensorflow_hub as tf_hub
import numpy as np
import cv2
import imageio
from tensorflow_docs.vis import embed
from matplotlib import pyplot as plt
from matplotlib.collections import LineCollection

movenet_lightning

In [2]:
model_lightning = tf_hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4")
size_lightning = 192

In [ ]:
# получаю предсказания модели на фотографии
def detection(image_path, model, input_size):
  image = tf.io.read_file(image_path)
  image = tf.image.decode_jpeg(image)
  # print(image.shape)
  img = tf.expand_dims(image, axis=0)
  resized_img = tf.image.resize_with_pad(img, input_size, input_size)
  img_np = resized_img.numpy().astype(np.int32)
  output = model.signatures["serving_default"](tf.constant(img_np))
  keypoints = output['output_0'].numpy()
  return keypoints, resized_img

input_image_path = '/content/frame_0016.jpg'
keypoints, resized_img = detection(input_image_path, model_lightning, size_lightning)
print(keypoints[0][0])

In [ ]:
points = ['nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear', 'left_shoulder', 'right_shoulder']
# цифры в connections - индексы в points, нос связан с глазами (0, 1) и (0, 2) и тд.
connections = [(0, 1), (0, 2), (1, 3), (2, 4), (0, 5), (0, 6), (5, 6)]
def make_skeleton(keypoints, image, image_height, image_width):
  num_people = keypoints.shape[0]
  fig, ax = plt.subplots(figsize=(12 * (float(image_width) / image_height), 12))
  scat = ax.scatter([], [], s=60, color='#FF1333', zorder=3)
  background_image = ax.imshow(image)
  arr_edges = []
  for i in range(num_people):
    seven_points = keypoints[i][0][:7]
    print(seven_points)
    x = [a[1] for a in seven_points]
    y = [a[0] for a in seven_points]
    arr_coordinates_x = [float(coordinate) * image_width for coordinate in x]
    arr_coordinates_y = [float(coordinate) * image_height for coordinate in y]
  arr_coord_point_1 = []
  arr_coord_point_2 = []
  for i in range(len(connections)):
    point_1 = connections[i][0]
    point_2 = connections[i][1]
    coord_point_1 = (round(arr_coordinates_x[point_1], 4), round(arr_coordinates_y[point_1], 4)) # кортеж координат первой точки
    coord_point_2 = (round(arr_coordinates_x[point_2], 4), round(arr_coordinates_y[point_2], 4)) # кортеж координат второй точки
    arr_coord_point_1.append(coord_point_1)
    arr_coord_point_2.append(coord_point_2)

  print(arr_coord_point_1)
  print(arr_coord_point_2)
  coord_points = np.stack([arr_coord_point_1, arr_coord_point_2], axis=1)
  coord_concat = np.concatenate(coord_points, axis=0)

  lines = np.array([arr_coord_point_1, arr_coord_point_2])
  merged_lines = np.stack(lines, axis=1)
  line_collection = LineCollection([], linewidths=(4), linestyle='solid')
  ax.add_collection(line_collection)
  line_collection.set_segments(merged_lines)
  scat.set_offsets(coord_concat)


# тестирование функции:
input_image = tf.io.read_file(input_image_path)
input_image = tf.image.decode_jpeg(input_image)
print(input_image.shape)
picture = tf.expand_dims(input_image, axis=0)
picture = tf.image.resize_with_pad(picture, size_lightning, size_lightning)
print(picture.shape)
display_pic = tf.expand_dims(input_image, axis=0)
display_pic = tf.cast(tf.image.resize_with_pad(display_pic, 1280, 1280), dtype=tf.int32)
result_image = np.squeeze(display_pic.numpy(), axis=0)
image_height, image_width, a = result_image.shape
print(image_height, image_width)

make_skeleton(keypoints, result_image, image_height, image_width)


Проверим исходные модели movenet_lightning на качество на датасете с сидячими людьми (80 фотографий).

Метрика PDJ: https://stasiuk.medium.com/pose-estimation-metrics-844c07ba0a78

In [39]:
## COCO FORMAT:
# 0 - nose,
# 1 - neck,
# 2 - right_shoulder,
# 3 - right_elbow,
# 4 - right_wrist,
# 5 - left_shoulder,
# 6 - left_elbow,
# 7 - left_wrist,
# 8 - right_hip,
# 9 - right_knee,
# 10 - right_ankle,
# 11 - left_hip,
# 12 - left_knee,
# 13 - left_ankle,
# 14 - right_eye,
# 15 - left_eye,
# 16 - right_ear,
# 17 - left_ear

# MoveNet FORMAT: ['nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear', 'left_shoulder', 'right_shoulder']

# проверка исходной модели на качество в разрезе нашей задачи:
import os
import json

val_dataset_path = '/content/drive/MyDrive/sitting_pose_dataset'
val_dataset_keypoints = []
sorted_dataset = []
for x in os.listdir(val_dataset_path):
  val_dataset_keypoints.append(x)
sorted_dataset = sorted(val_dataset_keypoints)

def make_predictions(model, model_size, dataset): # подаю датасет, получаю точки от модели
  val_dataset_keypoints = []
  pairs_xy = []
  num = 0
  for photo in dataset:
    image_path = '/content/drive/MyDrive/sitting_pose_dataset/' + str(photo)
    photo_keypoints, _ = detection(image_path, model, model_size)
    resized_x = photo_keypoints[0, 0, :, 1] * 1280
    resized_y = photo_keypoints[0, 0, :, 0] * 1280
    for i in range(len(resized_x)):
      pairs_xy.append((resized_x[i], resized_y[i]))
    val_dataset_keypoints.append((resized_x, resized_y))
    num += 1
  print(*[f"{np.float32(x)}, \n" for x in pairs_xy])


def extract_ideal_annotations(true_annotations): # подаю ground_truth аннотации из файла, получаю списки
                                                 # нужных точек
  with open("/content/result_TRUE80.json", "r", encoding="utf-8") as ground_truth:
    info = json.load(ground_truth)
  arr = []
  extracted_keypoints = []
  for x in info.get("annotations"):
    arr.append(x.get("keypoints"))
  for i in range(len(arr)):
    clean_arr = []
    if str(arr[i]) != "None":
      tmp = arr[i]
      clean_arr.append([tmp[0], tmp[1], tmp[2], # nose
                        tmp[45], tmp[46], tmp[47], # left_eye
                        tmp[42], tmp[43], tmp[44], # right_eye
                        tmp[51], tmp[52], tmp[53], # left_ear
                        tmp[48], tmp[49], tmp[50], # right_ear
                        tmp[15], tmp[16], tmp[17], # left_shoulder
                        tmp[6], tmp[7], tmp[8]] # right_shoulder
                       )
      extracted_keypoints.append(clean_arr)
  print(*[f"{x}, \n" for x in extracted_keypoints])

In [ ]:
make_predictions(model_lightning, size_lightning, sorted_dataset)

In [ ]:
extract_ideal_annotations("/content/result_TRUE80.json")

In [ ]:
# def compare_pred_true(predictions, ground_truth): # используется метрика PDJ - percentage of detected joints

In [ ]:
# пример того, как выглядит скелет на фотографии 720 * 1280 с координатами из идеальной разметки
def draw_image_ex(pic, image_height, image_width):
  fig, ax = plt.subplots(figsize=(12 * (float(image_width) / image_height), 12))
  scat = ax.scatter([], [], s=60, color='#FF1333', zorder=3)
  background_image = ax.imshow(image)
  arr_edges = []
  x = [610, 687, 565, 781, 504, 878, 371]
  y = [221, 187, 189, 254, 262, 480, 469]
  arr_coordinates_x = [float(coordinate) for coordinate in x]
  arr_coordinates_y = [float(coordinate) for coordinate in y]
  arr_coord_point_1 = []
  arr_coord_point_2 = []
  for i in range(len(connections)):
    point_1 = connections[i][0]
    point_2 = connections[i][1]
    coord_point_1 = (round(arr_coordinates_x[point_1], 4), round(arr_coordinates_y[point_1], 4)) # кортеж координат первой точки
    coord_point_2 = (round(arr_coordinates_x[point_2], 4), round(arr_coordinates_y[point_2], 4)) # кортеж координат второй точки
    arr_coord_point_1.append(coord_point_1)
    arr_coord_point_2.append(coord_point_2)

  print(arr_coord_point_1)
  print(arr_coord_point_2)
  coord_points = np.stack([arr_coord_point_1, arr_coord_point_2], axis=1)
  coord_concat = np.concatenate(coord_points, axis=0)

  lines = np.array([arr_coord_point_1, arr_coord_point_2])
  merged_lines = np.stack(lines, axis=1)
  line_collection = LineCollection([], linewidths=(4), linestyle='solid')
  ax.add_collection(line_collection)
  line_collection.set_segments(merged_lines)
  scat.set_offsets(coord_concat)

image = tf.io.read_file("/content/frame_0016.jpg")
image = tf.image.decode_jpeg(image)
image_height, image_width, alpha = image.shape
draw_image_ex(image, image_height, image_width)

movenet_thunder

In [6]:
model_thunder = tf_hub.load("https://tfhub.dev/google/movenet/singlepose/thunder/4")
size_thunder = 256

In [ ]:
keypoints_thunder, resized_img = detection(input_image_path, model_thunder, size_thunder)
print(keypoints_thunder)

In [ ]:
make_skeleton(keypoints_thunder, result_image, image_height, image_width)

In [ ]:
make_predictions(model_thunder, size_thunder, sorted_dataset)

Черновик:

Классификатор MLP на scikit-learn: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

def metrics_calculation(true, pred):
  metric = {'Accuracy': round(metrics.accuracy_score(true, pred), 3)}
  print("Accuracy: ", metric['Accuracy'])
  return metric

In [ ]:
sitting_dataset_path = '/content/drive/MyDrive/sitting_pose_dataset' # поменять путь, тк нужна ещё будет классификация
categories = ['correct', 'head_left', 'head_right', 'body_left', 'body_right', 'too_close', 'tilt_back', 'bend_over']

X, y = make_classification(n_samples=8, random_state=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1, test_size=0.3)
model = MLPClassifier(hidden_layer_sizes=(16, 32, 64, 128), # grid_search должен найти размер
                      alpha=0.01,
                      solver=lbfgs,
                      max_iter=300,
                      random_state=42)
model.fit(X_train, y_train)

In [ ]:
# находим метрику accuracy:
y_pred = model.predict(X_test)
metrics_calculation(y_test, y_pred)